# Problem Set · Case 4 · Should we badge more sellers? (Part II, individual)
**BUS-G 492 · Causal decisions with AI**

In [ ]:
# --- Setup: run this cell first -----------------------------------------------
import os
DATA_URL = "https://raw.githubusercontent.com/boyoung-seo/492-coding/main/cases/case4_panel_badge/data/"
for candidate in ["data/", "cases/case4_panel_badge/data/", "../cases/case4_panel_badge/data/", "../../../cases/case4_panel_badge/data/", "../../../../cases/case4_panel_badge/data/"]:
    if os.path.exists(candidate + "seller_months.csv"):
        DATA = candidate
        break
else:
    DATA = DATA_URL
print("Reading data from:", DATA)

## Something you didn't know

> **From:** Sara Lindqvist, Trust & Safety
> **Re:** how badges are actually granted
>
> We don't hand badges out at random. Our reviewers badge sellers they consider **excellent**: consistent quality, fast shipping, great service. Those sellers were usually doing very well **before** they got the badge.
>
> The **month** a seller gets the badge mostly depends on when a reviewer reaches their application in our **queue**, which can take anywhere from a few weeks to a year and has nothing to do with the seller's sales that month. Grant dates: `badge_grants.csv`.

**Use of AI:** strictly prohibited for questions with a * sign: **Q4, Q5, Q6, Q9**. Permitted for Q7 and Q8.

## Why Part I went wrong

Answer Q4–Q6 **before writing any code**, in your own words.

**Q4\* (2 points).** Why does the Trust & Safety email change how you read your Part 1 results? What were badged sellers like before their badge?

**Q4 answer** (double-click to type)



**Q5\* (3 points).** Explain why controlling for `avg_rating` does not solve the problem.

**Q5 answer** (double-click to type)



**Q6\* (2 points).** You have 36 months of data for every seller. How could you compare each seller **with itself**? What would that remove, and what would it still not remove? (Hint: think of things that hit every seller in the same month.)

**Q6 answer** (double-click to type)

- **Comparing each seller with itself removes:** 
- **It still does not remove:** 

## Redo the analysis

You may use AI for Q7 and Q8. This time **tell it the whole story**: paste the Trust & Safety email and say that the data has one row per seller per month.

### Load the data

This is a new notebook, so nothing from Part 1 is loaded yet. Run this cell to load the Part 1 data again, then the next cell for the new files.

In [ ]:
%pip install -q linearmodels   # not preinstalled in Colab
import statsmodels.formula.api as smf
import pandas as pd, numpy as np
df = pd.read_csv(DATA + "seller_months.csv")
print(df.shape)
df.head()

In [ ]:
grants = pd.read_csv(DATA + "badge_grants.csv")
grants.head()

### Step 1 · Were badged sellers already big before the badge?

Run the code below and look. It lines up every seller who got the badge **during** 2023–2025 by months relative to their badge month (`rel`: −1 = the month before, 0 = the badge month) and compares their revenue before and after the badge with sellers who were never badged. Nothing to report: compare the first number with the last, and use what you see in Q4.

In [ ]:
import numpy as np
g = grants[grants.badge_granted.notna() & (grants.badge_granted != "before 2023")].copy()
g["gm"] = pd.PeriodIndex(g.badge_granted, freq="M")
e = df.merge(g[["seller_id", "gm"]], on="seller_id")
e["rel"] = pd.PeriodIndex(e.month, freq="M").asi8 - pd.PeriodIndex(e.gm, freq="M").asi8
never = df[~df.seller_id.isin(grants[grants.badge_granted.notna()].seller_id)]
print(f"Badged sellers, 12 months before the badge: ${e[e.rel.between(-12, -1)].revenue.mean():,.0f} a month")
print(f"Badged sellers, 12 months after the badge:  ${e[e.rel.between(0, 11)].revenue.mean():,.0f} a month")
print(f"Sellers never badged (all months):          ${never.revenue.mean():,.0f} a month")


## Q7 (3 points) · Look back

### Step 2 · Compare each seller with itself (fixed effects) → ETT

Steps 1–3 use only the **2023–2025 data** to estimate what the badge did for the sellers who got it: the effect on the treated (ETT). Nothing about 2026 yet.

The code below estimates the badge effect on **log revenue** with **seller fixed effects** (each seller compared with itself) and **month fixed effects** (shocks that hit every seller in the same month), and prints `exp(coefficient) − 1`. Run it and report the estimate.

Optional: compare it with your Part 1 estimate. How much of the Part 1 effect was seller quality?

In [ ]:
from linearmodels.panel import PanelOLS
df["lr"] = np.log(df.revenue)
p = df.set_index(["seller_id", "month"])
p.index = p.index.set_levels([p.index.levels[0], pd.to_datetime(p.index.levels[1])])
fe = PanelOLS.from_formula("lr ~ has_badge + EntityEffects + TimeEffects", p).fit(cov_type="clustered", cluster_entity=True)
print(f"seller + month FE: {np.exp(fe.params['has_badge']) - 1:+.1%}")


### Step 3 · Check for a trend before the badge (event study)

The code below estimates the badge effect separately for each month from 12 before to 12 after the badge (−1 is the reference) and plots the coefficients. Run it and look at the plot.

Is there a trend before the badge? What would a rising trend before the badge mean for your Step 2 estimate?

In [ ]:
e2 = df.merge(g[["seller_id", "gm"]], on="seller_id", how="left")
rel = pd.PeriodIndex(e2.month, freq="M").asi8 - pd.PeriodIndex(e2.gm, freq="M").asi8
e2["rel"] = np.where(e2.gm.isna(), np.nan, np.clip(rel, -12, 12))
cols = []
for k in range(-12, 13):
    if k == -1:
        continue
    c = f"r{k + 12:02d}"; e2[c] = (e2.rel == k).astype(int); cols.append((k, c))
early = grants[grants.badge_granted == "before 2023"].seller_id
ep = e2[~e2.seller_id.isin(early)].set_index(["seller_id", "month"])
ep.index = ep.index.set_levels([ep.index.levels[0], pd.to_datetime(ep.index.levels[1])])
es = PanelOLS(ep.lr, ep[[c for _, c in cols]], entity_effects=True, time_effects=True).fit(cov_type="clustered", cluster_entity=True)
coefs = pd.Series({k: es.params[c] for k, c in cols})
coefs[-1] = 0.0
ax = coefs.sort_index().plot(marker="o", figsize=(8, 4))
ax.axhline(0, color="gray", lw=1); ax.axvline(-0.5, color="gray", ls="--", lw=1)
ax.set_xlabel("months relative to the badge"); ax.set_ylabel("effect on log revenue");


**Q7 answer** (double-click to type)

- **Fixed-effects badge effect on revenue (%):** 
- **Is there a trend before the badge, and what would a rising trend mean for the estimate?** 

## Q8 (2 points) · Look ahead: predict 2026 profit → choose R

Now **predict** what badging more sellers would earn in 2026: apply the effect you estimated in Step 2 to sellers who don't have the badge yet, using their 2025 revenue. This is a forecast: your deployment will show the actual 2026 result.

CraftHub earns **12%** of extra sales and pays **\$300 a year** per badge, so a badge pays for itself when 0.12 × ETT × 12 × monthly revenue is more than \$300.

Choose R. Predicted profit = Σ (0.12 × ETT × 12 × revenue − \$300) over unbadged sellers with 2025 average monthly revenue ≥ R in your sample, × 20 for all 40,000 sellers.

In [ ]:
# Your code here


**Q8 answer** (double-click to type)

- **Your R (\$):** 
- **Predicted 2026 profit for all 40,000 sellers (\$):** 

## Q9\* (3 points) · Recommendation

Write your recommendation to CraftHub's Head of Marketplace (five to eight sentences): the R you recommend and its predicted 2026 profit; why the comparison in Part I was misleading; and the key assumption your answer depends on.

**Q9 answer** (double-click to type)



---
**Before you submit:** choose Runtime → Restart session and run all, so every cell runs and shows its output. Then File → Download → Download .ipynb, named Lastname_Firstname_case4_part2.ipynb.

**Submit one zip file** containing two documents: your own Part II ipynb and your AI transcript with a signed statement.

- **AI transcript with a signed statement:** the complete conversation from every AI tool you used, as a share link or a full export (not screenshots or excerpts), and a signed statement that it is complete. If you used no AI tool, submit only the signed statement saying so. The AI submission is not graded and only used for assessing learning.